# CSA0812 – Assignment 5
## B3 – Loops vs Libraries vs SQL

**Student:** Darshan A  
**Register:** 1973260011  
**Marks:** 8 | **K4 – Analyse**

## B3 – Loops vs Libraries vs SQL (8 Marks)

This experiment compares three approaches on the same small dataset and then measures them at larger sizes:
1. Plain Python CSV loops using `statistics.mean()`.
2. pandas `groupby()`.
3. SQLite `GROUP BY`.

The small test must prove that the three methods return the same daily mean for each station/day key before performance results are compared.

### B3.1 – Correctness on Small Data (2 Marks)

The program computes daily mean PM2.5 using CSV loops, a running sum/count dictionary, pandas and SQL. Equality assertions compare the results. Any mismatch must be resolved before benchmarking.

### B3.2 – Increase the Data Size (2 Marks)

One year's hourly readings are generated for 1, 12 and 120 stations. Each station produces approximately 8,760 rows (365 days × 24 hours). The program records time for Python list-and-mean, running sum/count, pandas read/groupby, SQLite loading and SQL grouping.

The 120-station dataset contains 1,051,200 readings.

### B3.3 – Profile (2 Marks)

The plain-Python implementation is run under `cProfile`. The report is sorted by cumulative time to identify the function with the largest total contribution. In the prepared environment, the profiled `csv_loop_daily_means()` function was the main application function consuming the cumulative time; `statistics.mean()` was also visible among its costs. Actual ranking and timing can differ by runtime.

### B3.4 – Engineering Trade-Off (2 Marks)

For a quick one-time analysis of a small CSV, pandas is convenient and concise. For a daily automated report with repeated queries, SQLite is preferable because the data can be loaded once and aggregated repeatedly. A streaming running sum/count dictionary uses less memory than retaining every reading in lists.

The prepared environment measured:

| Stations | Rows | CSV lists + mean (s) | Running sum/count (s) | pandas (s) | SQL GROUP BY (s) | SQL load (s) |
|---:|---:|---:|---:|---:|---:|---:|
| 1 | 8,760 | See runtime output | See runtime output | See runtime output | See runtime output | See runtime output |
| 12 | 105,120 | 0.2292 | 0.1384 | 0.0328 | 0.0394 | 0.2058 |
| 120 | 1,051,200 | 1.9876 | 1.4560 | 0.3749 | 0.5063 | 2.0366 |

The timing values above are from one execution environment; use the values printed by your Colab session in the final report.

### Alternative Concept and Complexity Analysis

Instead of retaining every reading for a station/day and then calling `statistics.mean()`, keep a running sum and count for each key `(station, day)`.

```text
START
    CREATE empty dictionary TOTALS
    FOR each reading
        KEY = (station, day)
        IF KEY is not in TOTALS
            SET sum = 0 and count = 0
        ADD pm25 to sum
        ADD 1 to count
    END FOR
    FOR each KEY
        MEAN = sum / count
    END FOR
STOP
```

Both approaches process n input readings in O(n) time. The list-and-mean version stores O(n) readings. The running sum/count version stores O(stations × days) aggregate entries, which is smaller when many hourly readings belong to each station/day.

### Run the correctness, scale and profiling experiment

The code generates the datasets, checks equality of results, measures all three approaches and prints a cProfile report.

In [1]:
!pip -q install pandas

In [2]:
"""
Assignment 5 – B3: Loops vs Libraries vs SQL
Darshan A | 1973260011 | CSA0812

Run:
    python B3_loops_libraries_sql.py

This program:
1. Verifies daily PM2.5 means on a small dataset using CSV loops, pandas and SQL.
2. Generates one year of hourly data for 1, 12 and 120 stations (8,760 rows/station).
3. Measures each approach.
4. Profiles the plain-Python implementation with cProfile.
"""

from pathlib import Path
import csv
import sqlite3
import time
import statistics
import cProfile
import pstats
import io as string_io
from datetime import date, timedelta

import pandas as pd

OUT = Path("assignment5_b3_output")
OUT.mkdir(exist_ok=True)


def small_correctness_test():
    rows = [
        ("Adyar", "2026-10-01", 20.0),
        ("Adyar", "2026-10-01", 40.0),
        ("Guindy", "2026-10-01", 60.0),
        ("Adyar", "2026-10-02", 30.0),
        ("Guindy", "2026-10-02", 80.0),
    ]
    path = OUT / "small_readings.csv"
    with open(path, "w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(["station", "day", "pm25"])
        writer.writerows(rows)

    loop_result = csv_loop_daily_means(path)
    running_result = running_sum_daily_means(path)

    frame = pd.read_csv(path)
    pandas_result = {
        (row.station, row.day): round(row.pm25, 6)
        for row in frame.groupby(["station", "day"], as_index=False)["pm25"].mean().itertuples(index=False)
    }

    con = sqlite3.connect(":memory:")
    con.execute("CREATE TABLE aq(station TEXT, day TEXT, pm25 REAL)")
    con.executemany("INSERT INTO aq VALUES (?, ?, ?)", rows)
    sql_result = {
        (station, day): round(mean, 6)
        for station, day, mean in con.execute(
            "SELECT station, day, AVG(pm25) FROM aq GROUP BY station, day"
        )
    }
    con.close()

    assert loop_result == running_result == pandas_result == sql_result
    print("B3.1 SMALL-DATA CORRECTNESS")
    print("CSV loop + statistics.mean:", loop_result)
    print("Running sum/count dictionary:", running_result)
    print("pandas:", pandas_result)
    print("SQL:", sql_result)
    print("All methods agree.")


def csv_loop_daily_means(path):
    """Baseline loops: store every value for a station/day, then call statistics.mean."""
    values_by_station_day = {}
    with open(path, "r", newline="", encoding="utf-8") as file:
        reader = csv.DictReader(file)
        for row in reader:
            key = (row["station"], row["day"])
            values_by_station_day.setdefault(key, []).append(float(row["pm25"]))

    return {
        key: round(statistics.mean(values), 6)
        for key, values in values_by_station_day.items()
    }


def running_sum_daily_means(path):
    """Alternative: one pass with running sum/count; stores one pair per station/day."""
    totals = {}
    with open(path, "r", newline="", encoding="utf-8") as file:
        reader = csv.DictReader(file)
        for row in reader:
            key = (row["station"], row["day"])
            if key not in totals:
                totals[key] = [0.0, 0]
            totals[key][0] += float(row["pm25"])
            totals[key][1] += 1

    return {
        key: round(total / count, 6)
        for key, (total, count) in totals.items()
    }


def generate_year_csv(path, station_count):
    """Generate 365 days × 24 hourly observations for each station."""
    first_day = date(2026, 1, 1)
    rows_written = 0
    with open(path, "w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(["station", "day", "hour", "pm25"])
        for station_index in range(station_count):
            station = f"Station{station_index + 1:03d}"
            for day_offset in range(365):
                day = (first_day + timedelta(days=day_offset)).isoformat()
                for hour in range(24):
                    value = 20 + (station_index % 25) + (day_offset % 30) * 0.35 + hour * 0.2
                    writer.writerow([station, day, hour, round(value, 2)])
                    rows_written += 1
    return rows_written


def load_csv_to_sqlite(path, db_path):
    if db_path.exists():
        db_path.unlink()
    con = sqlite3.connect(db_path)
    con.execute("CREATE TABLE readings(station TEXT, day TEXT, hour INTEGER, pm25 REAL)")
    batch = []
    with open(path, "r", newline="", encoding="utf-8") as file:
        reader = csv.DictReader(file)
        for row in reader:
            batch.append((row["station"], row["day"], int(row["hour"]), float(row["pm25"])))
            if len(batch) >= 10000:
                con.executemany("INSERT INTO readings VALUES (?, ?, ?, ?)", batch)
                batch.clear()
    if batch:
        con.executemany("INSERT INTO readings VALUES (?, ?, ?, ?)", batch)
    con.commit()
    return con


def pandas_daily_means(path):
    frame = pd.read_csv(path)
    result = frame.groupby(["station", "day"], as_index=False)["pm25"].mean()
    return result


def sql_daily_means(con):
    return con.execute("""
        SELECT station, day, AVG(pm25) AS mean_pm25
        FROM readings
        GROUP BY station, day
    """).fetchall()


def profile_csv_loop(path):
    profiler = cProfile.Profile()
    profiler.enable()
    result = csv_loop_daily_means(path)
    profiler.disable()

    stream = string_io.StringIO()
    stats = pstats.Stats(profiler, stream=stream).sort_stats("cumulative")
    stats.print_stats(10)
    print("\nB3.3 cProfile – top cumulative-time functions")
    print(stream.getvalue())
    return result


def benchmark_station_sizes(station_counts=(1, 12, 120)):
    measurements = []
    print("\nB3.2 SCALE BENCHMARK")
    print("Stations | Rows | CSV lists+mean (s) | Running sum/count (s) | pandas read+groupby (s) | SQL GROUP BY (s) | SQL load (s)")

    for count in station_counts:
        csv_path = OUT / f"air_quality_{count}_stations.csv"
        db_path = OUT / f"air_quality_{count}_stations.db"

        rows = generate_year_csv(csv_path, count)

        start = time.perf_counter()
        loop_result = csv_loop_daily_means(csv_path)
        loop_seconds = time.perf_counter() - start

        start = time.perf_counter()
        running_result = running_sum_daily_means(csv_path)
        running_seconds = time.perf_counter() - start

        start = time.perf_counter()
        pandas_result = pandas_daily_means(csv_path)
        pandas_seconds = time.perf_counter() - start

        start = time.perf_counter()
        con = load_csv_to_sqlite(csv_path, db_path)
        load_seconds = time.perf_counter() - start

        start = time.perf_counter()
        sql_result = sql_daily_means(con)
        sql_seconds = time.perf_counter() - start

        # Correctness check on a handful of keys for every dataset size.
        pandas_dict = {
            (r.station, r.day): round(r.pm25, 6)
            for r in pandas_result.itertuples(index=False)
        }
        sql_dict = {
            (station, day): round(mean, 6)
            for station, day, mean in sql_result
        }
        check_keys = sorted(loop_result)[:10]
        loop_check = {key: loop_result[key] for key in check_keys}
        pandas_check = {key: pandas_dict[key] for key in check_keys}
        sql_check = {key: sql_dict[key] for key in check_keys}
        running_check = {key: running_result[key] for key in check_keys}
        assert loop_check == running_check == pandas_check == sql_check

        print(
            f"{count:8d} | {rows:10,d} | {loop_seconds:17.4f} | "
            f"{running_seconds:20.4f} | {pandas_seconds:22.4f} | "
            f"{sql_seconds:14.4f} | {load_seconds:10.4f}"
        )
        measurements.append({
            "stations": count,
            "rows": rows,
            "csv_lists_mean_seconds": loop_seconds,
            "running_sum_seconds": running_seconds,
            "pandas_seconds": pandas_seconds,
            "sql_group_seconds": sql_seconds,
            "sql_load_seconds": load_seconds,
        })

        con.close()

    return measurements


def main():
    small_correctness_test()

    measurements = benchmark_station_sizes()
    profile_path = OUT / "air_quality_12_stations.csv"
    profile_csv_loop(profile_path)
    output_csv = OUT / "benchmark_results.csv"
    with open(output_csv, "w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=list(measurements[0].keys()))
        writer.writeheader()
        writer.writerows(measurements)

    print("\nB3.4 ENGINEERING INTERPRETATION")
    print(
        "For a quick one-time analysis of a small CSV, pandas is convenient. "
        "For a daily automated report with repeated questions, SQLite is preferable "
        "because data can be stored once and queried repeatedly. The CSV loop is "
        "memory-efficient because it keeps running totals and counts instead of all rows."
    )
    print("Benchmark results saved to:", output_csv)
    print(
        "Complexity note: both Python methods require O(n) time. The list-and-mean version stores O(n) "
        "readings, while the running-sum/count version stores O(stations × days) aggregates."
    )
    print("B3 completed.")


if __name__ == "__main__":
    main()


B3.1 SMALL-DATA CORRECTNESS
CSV loop + statistics.mean: {('Adyar', '2026-10-01'): 30.0, ('Guindy', '2026-10-01'): 60.0, ('Adyar', '2026-10-02'): 30.0, ('Guindy', '2026-10-02'): 80.0}
Running sum/count dictionary: {('Adyar', '2026-10-01'): 30.0, ('Guindy', '2026-10-01'): 60.0, ('Adyar', '2026-10-02'): 30.0, ('Guindy', '2026-10-02'): 80.0}
pandas: {('Adyar', '2026-10-01'): 30.0, ('Adyar', '2026-10-02'): 30.0, ('Guindy', '2026-10-01'): 60.0, ('Guindy', '2026-10-02'): 80.0}
SQL: {('Adyar', '2026-10-01'): 30.0, ('Adyar', '2026-10-02'): 30.0, ('Guindy', '2026-10-01'): 60.0, ('Guindy', '2026-10-02'): 80.0}
All methods agree.

B3.2 SCALE BENCHMARK
Stations | Rows | CSV lists+mean (s) | Running sum/count (s) | pandas read+groupby (s) | SQL GROUP BY (s) | SQL load (s)
       1 |      8,760 |            0.0453 |               0.0296 |                 0.0116 |         0.0117 |     0.1004
      12 |    105,120 |            0.6201 |               0.2066 |                 0.0463 |         0.1369 |   

## Result and Interpretation

Check the printed equality statement, benchmark table and cProfile function report. Save the benchmark CSV and generated data files as evidence. Describe the speed/memory trade-off using measurements from your Colab run.